# Exploratory Data Analysis - 1
**Assignment Date:** 24 Mar  
**Platform:** PW Skills  
**Datasets Used:** Wine Quality Dataset, Student Performance Dataset


In [ ]:
# Common imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## Q1. What are the key features of the wine quality dataset? Discuss the importance of each feature in predicting the quality of wine.

### Answer:

The Wine Quality dataset (UCI ML Repository) contains **11 physicochemical input features** and **1 output variable (quality)**.

| Feature | Description | Importance in Predicting Quality |
|---|---|---|
| **fixed acidity** | Non-volatile acids (e.g., tartaric acid) | Contributes to tartness; higher acidity can lower perceived quality |
| **volatile acidity** | Amount of acetic acid (vinegar taste) | High levels cause unpleasant taste; **negatively** impacts quality |
| **citric acid** | Adds freshness and flavour | Small amounts improve quality; adds fruity notes |
| **residual sugar** | Sugar remaining after fermentation | Affects sweetness; very high = sweet wine, too high = poor quality dry wine |
| **chlorides** | Amount of salt | High chlorides make wine taste salty/bitter — **negatively** impacts quality |
| **free sulfur dioxide** | SO₂ not bound to other molecules | Protects wine from microbial spoilage and oxidation |
| **total sulfur dioxide** | Total SO₂ (free + bound) | Too high is detectable in smell/taste; regulated for quality |
| **density** | Mass per unit volume | Related to sugar & alcohol content; denser wines tend to have more sugar |
| **pH** | Measure of acidity/basicity | Affects microbial stability; typical range 3–4 for wine |
| **sulphates** | Additive contributing to SO₂ | Acts as antimicrobial; moderate levels **positively** correlate with quality |
| **alcohol** | % alcohol by volume | **Strongest positive correlator** with quality; higher alcohol = better quality (in general) |
| **quality** (target) | Score between 0–10 | Output variable based on sensory evaluation |

**Most important features:** `alcohol`, `volatile acidity`, `sulphates`, and `citric acid` are typically the strongest predictors of wine quality.


In [ ]:
# Load Wine Quality Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = pd.read_csv(url, sep=';')

print("Shape:", wine_df.shape)
print("\nColumns:", wine_df.columns.tolist())
print("\nFirst 5 rows:")
wine_df.head()

In [ ]:
# Correlation of features with quality
corr = wine_df.corr()['quality'].sort_values(ascending=False)
print("Correlation with Quality:")
print(corr)

plt.figure(figsize=(8, 5))
corr.drop('quality').plot(kind='bar', color='steelblue')
plt.title('Feature Correlation with Wine Quality')
plt.ylabel('Correlation Coefficient')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Q2. How did you handle missing data in the wine quality dataset during the feature engineering process? Discuss the advantages and disadvantages of different imputation techniques.

### Answer:

**Step 1: Check for missing data**


In [ ]:
print("Missing values in Wine Quality Dataset:")
print(wine_df.isnull().sum())
print("\nTotal missing values:", wine_df.isnull().sum().sum())

### Imputation Techniques — Advantages & Disadvantages:

| Technique | Description | Advantages | Disadvantages |
|---|---|---|---|
| **Mean Imputation** | Replace missing with column mean | Simple, fast | Distorts distribution; ignores relationships |
| **Median Imputation** | Replace with median | Robust to outliers | Ignores feature correlations |
| **Mode Imputation** | Replace with most frequent value | Good for categorical | Can create bias |
| **KNN Imputation** | Use K nearest neighbours to fill | Considers feature relationships | Computationally expensive |
| **Iterative Imputation (MICE)** | Models each feature as function of others | Most accurate | Slow; complex to implement |
| **Drop rows** | Remove rows with missing values | Simple; preserves distribution | Loses data; risky for small datasets |
| **Forward/Backward Fill** | Use previous/next value | Good for time series | Not suitable for non-sequential data |


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer
import pandas as pd
import numpy as np

# Create a copy with artificially introduced missing values for demo
wine_missing = wine_df.copy()
np.random.seed(42)
for col in ['alcohol', 'pH', 'sulphates']:
    wine_missing.loc[wine_missing.sample(frac=0.05).index, col] = np.nan

print("Introduced missing values:")
print(wine_missing.isnull().sum()[wine_missing.isnull().sum() > 0])

# Mean Imputation
mean_imputer = SimpleImputer(strategy='mean')
wine_mean_imputed = pd.DataFrame(mean_imputer.fit_transform(wine_missing), columns=wine_missing.columns)
print("\nMean Imputation done. Missing values remaining:", wine_mean_imputed.isnull().sum().sum())

# KNN Imputation
knn_imputer = KNNImputer(n_neighbors=5)
wine_knn_imputed = pd.DataFrame(knn_imputer.fit_transform(wine_missing), columns=wine_missing.columns)
print("KNN Imputation done. Missing values remaining:", wine_knn_imputed.isnull().sum().sum())

## Q3. What are the key factors that affect students' performance in exams? How would you go about analyzing these factors using statistical techniques?

### Answer:

**Key Factors Affecting Student Performance:**

1. **Parental Education Level** — Higher parental education → better academic support at home
2. **Test Preparation Course** — Students who complete prep courses tend to score higher
3. **Gender** — Slight performance differences across subjects
4. **Lunch Type** — Free/reduced lunch is a proxy for socioeconomic status
5. **Race/Ethnicity** — Captures socioeconomic and cultural background
6. **Reading & Writing Scores** — Strongly correlated with math performance

**Statistical Techniques for Analysis:**

| Technique | Purpose |
|---|---|
| **Descriptive Statistics** | Mean, median, std dev of scores per group |
| **Correlation Analysis** | Pearson/Spearman correlation between scores |
| **T-test** | Compare means of two groups (e.g., test prep: yes vs no) |
| **ANOVA** | Compare means across multiple groups (e.g., parental education levels) |
| **Chi-square Test** | Association between categorical variables |
| **Regression Analysis** | Quantify effect size of each factor on score |


In [ ]:
# Load Student Performance Dataset
stud_url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/StudentsPerformance.csv"
stud_df = pd.read_csv(stud_url)

print("Shape:", stud_df.shape)
print("\nColumns:", stud_df.columns.tolist())
stud_df.head()

In [ ]:
# Add average score column
stud_df['average score'] = stud_df[['math score', 'reading score', 'writing score']].mean(axis=1)

# T-test: Impact of test preparation course
completed = stud_df[stud_df['test preparation course'] == 'completed']['average score']
none_prep = stud_df[stud_df['test preparation course'] == 'none']['average score']

t_stat, p_val = stats.ttest_ind(completed, none_prep)
print(f"T-test (Test Prep vs No Prep):")
print(f"  Mean (completed): {completed.mean():.2f}")
print(f"  Mean (none): {none_prep.mean():.2f}")
print(f"  T-statistic: {t_stat:.4f}, P-value: {p_val:.4f}")
print(f"  Conclusion: {'Significant difference' if p_val < 0.05 else 'No significant difference'}")

# Visualization
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
stud_df.groupby('test preparation course')['average score'].mean().plot(kind='bar', color=['coral', 'steelblue'])
plt.title('Avg Score by Test Prep')
plt.ylabel('Average Score')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
stud_df.groupby('parental level of education')['average score'].mean().sort_values().plot(kind='barh', color='mediumseagreen')
plt.title('Avg Score by Parental Education')
plt.tight_layout()
plt.show()

## Q4. Describe the process of feature engineering in the context of the student performance dataset. How did you select and transform the variables for your model?

### Answer:

**Feature Engineering** is the process of using domain knowledge to create, transform, or select features to improve model performance.

**Steps performed:**
1. **Creating new features** — Average score, total score, pass/fail flag
2. **Encoding categorical variables** — Label Encoding / One-Hot Encoding
3. **Handling missing values** — Imputation
4. **Feature selection** — Correlation, feature importance
5. **Scaling** — StandardScaler / MinMaxScaler for numeric features


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

stud_fe = stud_df.copy()

# 1. New Features
stud_fe['total score'] = stud_fe['math score'] + stud_fe['reading score'] + stud_fe['writing score']
stud_fe['pass_fail'] = (stud_fe['average score'] >= 50).astype(int)  # 1=Pass, 0=Fail

print("New features added: 'total score', 'pass_fail'")
print(stud_fe[['math score', 'reading score', 'writing score', 'average score', 'total score', 'pass_fail']].head())

# 2. Label Encoding for categorical columns
cat_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
le = LabelEncoder()
for col in cat_cols:
    stud_fe[col + '_encoded'] = le.fit_transform(stud_fe[col])

print("\nEncoded columns added.")

# 3. Feature Selection via Correlation
encoded_cols = [c for c in stud_fe.columns if '_encoded' in c]
feature_cols = encoded_cols + ['math score', 'reading score', 'writing score']
corr_matrix = stud_fe[feature_cols + ['average score']].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap — Student Performance Features')
plt.tight_layout()
plt.show()

# 4. Scaling numeric features
scaler = StandardScaler()
numeric_cols = ['math score', 'reading score', 'writing score']
stud_fe[['math_scaled', 'reading_scaled', 'writing_scaled']] = scaler.fit_transform(stud_fe[numeric_cols])
print("\nScaled features sample:")
print(stud_fe[['math_scaled', 'reading_scaled', 'writing_scaled']].head())

## Q5. Load the wine quality dataset and perform EDA to identify the distribution of each feature. Which features exhibit non-normality, and what transformations could be applied to improve normality?

In [ ]:
# Distribution plots for all features
features = wine_df.columns.drop('quality')

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(wine_df[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distribution of Each Feature — Wine Quality Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Shapiro-Wilk Normality Test for each feature
from scipy.stats import shapiro

print(f"{'Feature':<30} {'W-Stat':>10} {'P-Value':>12} {'Normal?':>10}")
print("-" * 65)
for col in features:
    stat, p = shapiro(wine_df[col].sample(500, random_state=42))  # sample for speed
    normal = 'Yes' if p > 0.05 else 'No'
    print(f"{col:<30} {stat:>10.4f} {p:>12.4f} {normal:>10}")

### Features with Non-Normality & Suggested Transformations:

| Feature | Skewness Type | Transformation |
|---|---|---|
| `residual sugar` | Right-skewed (positive) | Log Transform |
| `chlorides` | Right-skewed | Log Transform |
| `free sulfur dioxide` | Right-skewed | Square Root / Log |
| `total sulfur dioxide` | Right-skewed | Log Transform |
| `volatile acidity` | Slightly skewed | Square Root |
| `citric acid` | Bimodal / skewed | Log or Box-Cox |


In [ ]:
from scipy.stats import boxcox

# Apply Log Transformation to right-skewed features
skewed_features = ['residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide']

fig, axes = plt.subplots(len(skewed_features), 2, figsize=(12, 14))

for i, col in enumerate(skewed_features):
    # Original
    axes[i, 0].hist(wine_df[col], bins=30, color='salmon', edgecolor='white')
    axes[i, 0].set_title(f'{col} — Original')

    # Log transformed
    log_transformed = np.log1p(wine_df[col])
    axes[i, 1].hist(log_transformed, bins=30, color='mediumseagreen', edgecolor='white')
    axes[i, 1].set_title(f'{col} — Log Transformed')

plt.suptitle('Before vs After Log Transformation', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Skewness comparison:")
for col in skewed_features:
    orig_skew = wine_df[col].skew()
    log_skew = np.log1p(wine_df[col]).skew()
    print(f"{col:<30} Original: {orig_skew:>6.3f}   Log: {log_skew:>6.3f}")

## Q6. Using the wine quality dataset, perform Principal Component Analysis (PCA) to reduce the number of features. What is the minimum number of principal components required to explain 90% of the variance in the data?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Step 1: Separate features and scale
X = wine_df.drop('quality', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Apply PCA with all components
pca = PCA()
pca.fit(X_scaled)

# Step 3: Explained variance
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("Individual Explained Variance per Component:")
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var)):
    print(f"  PC{i+1}: {ev*100:.2f}%   Cumulative: {cv*100:.2f}%")

# Step 4: Minimum components for 90% variance
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\n✅ Minimum components to explain 90% variance: {n_components_90}")

In [ ]:
# Visualization: Scree Plot + Cumulative Variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree Plot
axes[0].bar(range(1, len(explained_var)+1), explained_var * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Scree Plot — Explained Variance per PC')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_xticks(range(1, len(explained_var)+1))

# Cumulative Variance Plot
axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var * 100, marker='o', color='darkorange')
axes[1].axhline(y=90, color='red', linestyle='--', label='90% threshold')
axes[1].axvline(x=n_components_90, color='green', linestyle='--', label=f'PC {n_components_90}')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of Principal Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xticks(range(1, len(cumulative_var)+1))
axes[1].legend()

plt.suptitle('PCA Analysis — Wine Quality Dataset', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nConclusion: {n_components_90} principal components out of {X.shape[1]} original features")
print(f"are sufficient to explain at least 90% of the variance in the Wine Quality dataset.")

In [ ]:
# Apply PCA with optimal number of components
pca_final = PCA(n_components=n_components_90)
X_pca = pca_final.fit_transform(X_scaled)

print(f"Original feature matrix shape: {X_scaled.shape}")
print(f"Reduced feature matrix shape:  {X_pca.shape}")
print(f"Variance retained: {pca_final.explained_variance_ratio_.sum()*100:.2f}%")